# Machine Learning – Practical 5: Support Vector Machine (Linear vs RBF) on Iris

**Aim:** To classify Iris flowers using SVM and compare Linear and RBF kernels.

**Objective:**
- Train Linear and RBF SVM classifiers
- Evaluate with accuracy, precision, recall and F1
- Compare confusion matrices

**Theory:** SVM finds the hyperplane that separates classes with the maximum margin. A Linear kernel uses a straight boundary; the RBF kernel maps data to a higher dimension to learn non-linear boundaries. `C` controls margin vs misclassification trade-off.

## 1. Import Libraries

**Aim:** Import required tools.

**Objective:** Load data, model, pipeline and metric libraries.

**Theory:** scikit-learn provides the SVC classifier, Pipeline for chaining steps and metrics for evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("All libraries imported successfully!")

## 2. Load the Iris Dataset

**Aim:** Load the dataset.

**Objective:** Get features, target and class names.

**Theory:** Iris has 150 samples, 4 features (sepal/petal length and width) and 3 classes: setosa, versicolor, virginica.

In [ ]:
iris = load_iris()

X = iris.data
y = iris.target
class_names = iris.target_names

print("Dataset: Iris")
print("Total samples:", X.shape[0])
print("Number of features:", X.shape[1])
print("Classes:", list(class_names))

## 3. Train-Test Split

**Aim:** Create training and testing sets.

**Objective:** Split 80/20 with stratification.

**Theory:** Stratification keeps equal class proportions in both sets, which matters for small datasets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 4. Define SVM Models

**Aim:** Create the two kernels to compare.

**Objective:** Define Linear SVM and RBF SVM.

**Theory:** Linear works well when classes are linearly separable; RBF (`gamma='scale'`) handles curved boundaries.

In [ ]:
models = {
    "Linear SVM": SVC(kernel="linear", C=1.0),
    "RBF SVM": SVC(kernel="rbf", C=1.0, gamma="scale")
}

results = []
confusion_matrices = {}

print("Linear and RBF SVM models created successfully!")

## 5. Train & Evaluate

**Aim:** Fit each model and measure performance.

**Objective:** Scale features, train, predict and compute macro metrics.

**Theory:** SVM is distance-based, so features must be scaled. Macro averaging gives each class equal weight.

In [ ]:
for model_name, classifier in models.items():

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", classifier)
    ])

    # Train the model
    model.fit(X_train, y_train)

    # Predict test labels
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_test, y_pred,
        average="macro",
        zero_division=0
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1-Score": f1
    })

    confusion_matrices[model_name] = confusion_matrix(
        y_test, y_pred
    )

    print("\n" + "=" * 55)
    print(model_name)
    print("=" * 55)
    print(f"Accuracy        : {accuracy:.4f}")
    print(f"Macro Precision : {precision:.4f}")
    print(f"Macro Recall    : {recall:.4f}")
    print(f"Macro F1-Score  : {f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        zero_division=0
    ))

## 6. Performance Comparison

**Aim:** Compare the two kernels.

**Objective:** Tabulate all metrics side by side.

**Theory:** A table makes it easy to see which kernel generalises better on this dataset.

In [ ]:
results_df = pd.DataFrame(results).set_index("Model")

print("Performance Comparison:")
print(results_df.round(4).to_string())

## 7. Confusion Matrices

**Aim:** Visualise misclassifications.

**Objective:** Plot a heatmap for each model.

**Theory:** Diagonal cells are correct predictions; off-diagonal cells show which classes get confused.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (model_name, cm) in zip(
    axes, confusion_matrices.items()
):
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax
    )

    ax.set_title(model_name)
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("Actual Class")

plt.tight_layout()
plt.show()

## 8. Best Model

**Aim:** Pick the better model.

**Objective:** Select the model with the highest macro F1-score.

**Theory:** Macro F1 balances precision and recall across all classes.

In [ ]:
best_model = results_df["Macro F1-Score"].idxmax()

print(
    "\nModel with the highest test macro F1-score:",
    best_model
)

print(
    "Note: This comparison uses one stratified train-test split."
)

## Conclusion

Linear SVM scored 100% accuracy and RBF SVM 96.67% (one versicolor misclassified as virginica). With only 30 test samples, the difference is small; cross-validation would give a more reliable comparison.